# Ethereum AML GNN Visualization Pipeline (Production)

**Purpose:** Generate five complementary visualizations from trained NodeGINe GNN model on Ethereum OFAC sanctions detection task.

**Data:** Real OFAC-sanctioned Ethereum addresses + EOA-only transaction graph (~889K nodes, ~23M edges)

**Output:** Publication-ready figures demonstrating:
1. UMAP embedding projections (representation quality)
2. Ego-centric subgraph motifs (transaction patterns)
3. Temporal evolution snapshots (temporal dynamics)
4. SHAP explainability (model decisions)
5. Evaluation metrics (confusion matrix, ROC/PR curves)
---

## Cell 1: Install Dependencies & Imports

In [1]:
# Install required packages (Colab-compatible)
import subprocess
import sys

packages = [
    'torch',
    'torch-geometric',
    'umap-learn',
    'pandas',
    'pyarrow',
    'networkx',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'pyvis',
    'shap'
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ All dependencies installed")

✓ All dependencies installed


In [2]:
# Core imports
import os
import sys
import logging
import warnings
from pathlib import Path
from typing import Tuple, Dict, List, Optional
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GINEConv, BatchNorm, Linear

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc, precision_recall_curve,
    f1_score, precision_score, recall_score
)
import networkx as nx
from pyvis.network import Network
import umap
import shap

# Suppress warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Matplotlib settings
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
sns.set_style('whitegrid')

print("✓ All imports loaded")

✓ All imports loaded


In [3]:
# Mount Google Drive (required for data access in Colab)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 2: Configuration & Logging

In [4]:
# Setup logging
def setup_logger(name: str) -> logging.Logger:
    """Configure logger with console output."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)

    # Remove existing handlers to avoid duplicates
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_format = logging.Formatter(
        '%(asctime)s [%(levelname)s] %(name)s: %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    console_handler.setFormatter(console_format)
    logger.addHandler(console_handler)
    return logger

logger = setup_logger('AML_Visualizations_Production')

# Configuration
class Config:
    """Data paths and hyperparameters."""

    # Base path for your project within Google Drive
    BASE_DRIVE_PATH = Path("/content/drive/MyDrive/Math_168_Folder")

    # Data directories
    DATA_DIR = BASE_DRIVE_PATH / "data" / "aggressively_processed"
    TRAINING_RESULTS_DIR = BASE_DRIVE_PATH / "training_results"
    OUTPUT_DIR = BASE_DRIVE_PATH / "Data-Visualization" / "outputs"

    # Input files (REAL DATA ONLY)
    FORMATTED_TRANSACTIONS = DATA_DIR / "formatted_transactions.parquet"
    NODE_LABELS = DATA_DIR / "node_labels.parquet"
    DATA_SPLITS = DATA_DIR / "data_splits.json"
    BEST_MODEL_CHECKPOINT = TRAINING_RESULTS_DIR / "best_model.pt"

    # Visualization hyperparameters
    UMAP_NEIGHBORS = 15
    UMAP_MIN_DIST = 0.1
    UMAP_N_EPOCHS = 200

    EMBEDDING_SUBSAMPLE_SIZE = 2000  # For visualization clarity
    EGO_SUBGRAPH_RADIUS = 2
    EGO_SUBGRAPH_MAX_SAMPLES = 4
    TEMPORAL_WINDOWS = 10

    SHAP_SAMPLE_SIZE = 100
    SHAP_BACKGROUND_SIZE = 50

    # Output settings
    DPI = 300
    FIGURE_FORMAT = 'png'

    @classmethod
    def validate(cls) -> bool:
        """Validate required files exist."""
        required = [cls.FORMATTED_TRANSACTIONS, cls.NODE_LABELS, cls.DATA_SPLITS, cls.BEST_MODEL_CHECKPOINT]
        missing = [f for f in required if not f.exists()]
        if missing:
            logger.error(f"Missing files: {missing}")
            return False
        cls.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        logger.info(f"✓ All files found. Output: {cls.OUTPUT_DIR}")
        return True

config = Config()
if not config.validate():
    raise RuntimeError("Configuration validation failed.")

2026-03-06 03:20:24 [INFO] AML_Visualizations_Production: ✓ All files found. Output: /content/drive/MyDrive/Math_168_Folder/Data-Visualization/outputs


INFO:AML_Visualizations_Production:✓ All files found. Output: /content/drive/MyDrive/Math_168_Folder/Data-Visualization/outputs


## Cell 3: Model Architecture

In [5]:
class NodeGINe(nn.Module):
    """GINe adapted for node-level classification."""

    def __init__(self, num_features, num_gnn_layers, n_classes=2,
                 n_hidden=100, edge_updates=False, residual=True,
                 edge_dim=None, dropout=0.0, final_dropout=0.5):
        super().__init__()
        self.n_hidden = n_hidden
        self.num_gnn_layers = num_gnn_layers
        self.edge_updates = edge_updates

        self.node_emb = nn.Linear(num_features, n_hidden)
        self.edge_emb = nn.Linear(edge_dim, n_hidden)

        self.convs = nn.ModuleList()
        self.emlps = nn.ModuleList()
        self.batch_norms = nn.ModuleList()

        for _ in range(self.num_gnn_layers):
            conv = GINEConv(nn.Sequential(
                nn.Linear(n_hidden, n_hidden),
                nn.ReLU(),
                nn.Linear(n_hidden, n_hidden)
            ), edge_dim=n_hidden)
            if self.edge_updates:
                self.emlps.append(nn.Sequential(
                    nn.Linear(3 * n_hidden, n_hidden),
                    nn.ReLU(),
                    nn.Linear(n_hidden, n_hidden),
                ))
            self.convs.append(conv)
            self.batch_norms.append(BatchNorm(n_hidden))

        self.mlp = nn.Sequential(
            Linear(n_hidden, 50), nn.ReLU(), nn.Dropout(final_dropout),
            Linear(50, 25), nn.ReLU(), nn.Dropout(final_dropout),
            Linear(25, n_classes)
        )

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)

        for i in range(self.num_gnn_layers):
            x = (x + F.relu(self.batch_norms[i](self.convs[i](x, edge_index, edge_attr)))) / 2
            if self.edge_updates:
                edge_attr = edge_attr + self.emlps[i](
                    torch.cat([x[src], x[dst], edge_attr], dim=-1)
                ) / 2

        return self.mlp(x)

    def get_embeddings(self, x, edge_index, edge_attr):
        """Extract node embeddings before MLP head."""
        src, dst = edge_index
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)

        for i in range(self.num_gnn_layers):
            x = (x + F.relu(self.batch_norms[i](self.convs[i](x, edge_index, edge_attr)))) / 2
            if self.edge_updates:
                edge_attr = edge_attr + self.emlps[i](
                    torch.cat([x[src], x[dst], edge_attr], dim=-1)
                ) / 2

        return x

logger.info("✓ NodeGINe model class loaded")

2026-03-06 03:20:24 [INFO] AML_Visualizations_Production: ✓ NodeGINe model class loaded


INFO:AML_Visualizations_Production:✓ NodeGINe model class loaded


## Cell 4: Data Loading & Preparation

In [6]:
def z_norm(data):
    """Z-score normalization."""
    std = data.std(0).unsqueeze(0)
    std = torch.where(std == 0, torch.tensor(1.0), std)
    return (data - data.mean(0).unsqueeze(0)) / std

def build_pyg_data(df_edges, df_labels, data_splits):
    """Build PyG Data object from real formatted data."""
    logger.info("Building PyG Data object from real data...")

    src = torch.LongTensor(df_edges['from_id'].values)
    dst = torch.LongTensor(df_edges['to_id'].values)
    edge_index = torch.stack([src, dst], dim=0)

    edge_feat_cols = ['EdgeID', 'Timestamp', 'Amount Sent', 'Sent Currency',
                      'Amount Received', 'Received Currency', 'Payment Format', 'Is Laundering']
    edge_feat_df = df_edges[edge_feat_cols].copy()
    edge_attr = torch.tensor(edge_feat_df.values, dtype=torch.float32)

    max_node_id = max(int(src.max()), int(dst.max())) + 1
    x = torch.ones(max_node_id, 1, dtype=torch.float32)

    y = torch.zeros(max_node_id, dtype=torch.long)
    for nid, lbl in zip(df_labels['node_id'].values, df_labels['is_sanctioned'].values):
        if nid < max_node_id:
            y[int(nid)] = int(lbl)

    train_mask = torch.zeros(max_node_id, dtype=torch.bool)
    val_mask = torch.zeros(max_node_id, dtype=torch.bool)

    if 'train_edge_ids' in data_splits:
        train_edge_ids = data_splits['train_edge_ids']
        train_edges_df = df_edges.iloc[train_edge_ids]
        train_nodes = set(train_edges_df['from_id'].values) | set(train_edges_df['to_id'].values)
        for nid in train_nodes:
            if nid < max_node_id:
                train_mask[int(nid)] = True

    if 'val_edge_ids' in data_splits:
        val_edge_ids = data_splits['val_edge_ids']
        val_edges_df = df_edges.iloc[val_edge_ids]
        val_nodes = set(val_edges_df['from_id'].values) | set(val_edges_df['to_id'].values)
        for nid in val_nodes:
            if nid < max_node_id:
                val_mask[int(nid)] = True
        val_mask = val_mask & ~train_mask

    edge_attr = z_norm(edge_attr)
    x = z_norm(x)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y,
                train_mask=train_mask, val_mask=val_mask)

    logger.info(f"  Nodes: {data.num_nodes:,}, Edges: {data.num_edges:,}")
    logger.info(f"  Sanctioned: {(data.y == 1).sum().item()}, "
                f"Train: {data.train_mask.sum().item():,}, "
                f"Val: {data.val_mask.sum().item():,}")

    return data

# Load real data
logger.info("[LOADING DATA]")
df_edges = pd.read_parquet(config.FORMATTED_TRANSACTIONS)
df_labels = pd.read_parquet(config.NODE_LABELS)
with open(config.DATA_SPLITS) as f:
    data_splits = json.load(f)

logger.info(f"  Loaded {len(df_edges):,} edges")
logger.info(f"  Loaded {len(df_labels):,} node labels")
logger.info(f"  Sanctioned nodes: {(df_labels['is_sanctioned'] == 1).sum()}")

# Build PyG Data
data = build_pyg_data(df_edges, df_labels, data_splits)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)
logger.info(f"✓ Device: {device}")

2026-03-06 03:20:24 [INFO] AML_Visualizations_Production: [LOADING DATA]


INFO:AML_Visualizations_Production:[LOADING DATA]


2026-03-06 03:20:40 [INFO] AML_Visualizations_Production:   Loaded 23,082,561 edges


INFO:AML_Visualizations_Production:  Loaded 23,082,561 edges


2026-03-06 03:20:40 [INFO] AML_Visualizations_Production:   Loaded 889,615 node labels


...

INFO:AML_Visualizations_Production:  Nodes: 889,615, Edges: 23,082,561


2026-03-06 03:21:04 [INFO] AML_Visualizations_Production:   Sanctioned: 69, Train: 777,754, Val: 111,861


INFO:AML_Visualizations_Production:  Sanctioned: 69, Train: 777,754, Val: 111,861


2026-03-06 03:21:04 [INFO] AML_Visualizations_Production: ✓ Device: cpu


INFO:AML_Visualizations_Production:✓ Device: cpu


## Cell 5: Load Trained Model

In [7]:
logger.info("[LOADING MODEL]")
checkpoint = torch.load(config.BEST_MODEL_CHECKPOINT, map_location=device)

model = NodeGINe(
    num_features=1,
    num_gnn_layers=2,
    n_classes=2,
    n_hidden=66,
    edge_updates=False,
    edge_dim=8,
    dropout=0.01,
    final_dropout=0.1
)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

logger.info(f"  Loaded from epoch {checkpoint['epoch'] + 1}")
logger.info(f"  Best Val F1: {checkpoint['val_f1']:.4f}")
logger.info(f"✓ Model loaded and ready")

2026-03-06 03:21:04 [INFO] AML_Visualizations_Production: [LOADING MODEL]


INFO:AML_Visualizations_Production:[LOADING MODEL]


2026-03-06 03:21:05 [INFO] AML_Visualizations_Production:   Loaded from epoch 78


INFO:AML_Visualizations_Production:  Loaded from epoch 78


2026-03-06 03:21:05 [INFO] AML_Visualizations_Production:   Best Val F1: 0.2414


INFO:AML_Visualizations_Production:  Best Val F1: 0.2414


2026-03-06 03:21:05 [INFO] AML_Visualizations_Production: ✓ Model loaded and ready


INFO:AML_Visualizations_Production:✓ Model loaded and ready


## Visualization 1: UMAP Embeddings (Production)

In [8]:
logger.info("\n[VIZ 1] UMAP Embedding Projection")

# Extract embeddings from validation set
logger.info("  Extracting node embeddings from trained model...")
with torch.no_grad():
    embeddings_full = model.get_embeddings(
        data.x, data.edge_index, data.edge_attr
    ).cpu().numpy()

# Use ALL validation nodes (no subsampling)
val_indices = torch.nonzero(data.val_mask, as_tuple=True)[0].numpy()
embeddings = embeddings_full[val_indices]
labels = data.y[val_indices].cpu().numpy()

n_sanctioned = (labels == 1).sum()
n_benign = (labels == 0).sum()
logger.info(f"  Using all {len(val_indices):,} validation nodes ({n_sanctioned} sanctioned, {n_benign:,} benign)")

# UMAP projection
logger.info("  Computing UMAP projection (this may take 3-6 minutes)...")
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

reducer = umap.UMAP(
    n_neighbors=config.UMAP_NEIGHBORS,
    min_dist=config.UMAP_MIN_DIST,
    metric='euclidean',
    n_epochs=config.UMAP_N_EPOCHS,
    random_state=42,
    verbose=1
)
embeddings_2d = reducer.fit_transform(embeddings_scaled)

# Plot — benign with low alpha to show density, sanctioned prominent on top
fig, ax = plt.subplots(figsize=(14, 11))

mask_benign = labels == 0
ax.scatter(embeddings_2d[mask_benign, 0], embeddings_2d[mask_benign, 1],
           c='steelblue', alpha=0.15, s=10, label=f'Non-sanctioned (n={mask_benign.sum():,})')

mask_sanctioned = labels == 1
ax.scatter(embeddings_2d[mask_sanctioned, 0], embeddings_2d[mask_sanctioned, 1],
           c='darkred', alpha=0.95, s=300, marker='*',
           label=f'OFAC-sanctioned (n={mask_sanctioned.sum()})',
           edgecolors='darkred', linewidths=2, zorder=10)

ax.set_xlabel('UMAP Dimension 1', fontsize=13, fontweight='bold')
ax.set_ylabel('UMAP Dimension 2', fontsize=13, fontweight='bold')
ax.set_title('Node Embedding Projections: GNN Learned Representations\n' +
             f'(UMAP 2D projection, all {len(val_indices):,} validation nodes)',
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.2, linestyle='--')

plt.tight_layout()
output_path = config.OUTPUT_DIR / 'viz1_embedding_projection_umap.png'
fig.savefig(output_path, dpi=config.DPI, bbox_inches='tight')
logger.info(f"  ✓ Saved: {output_path}")
plt.close()

2026-03-06 03:21:05 [INFO] AML_Visualizations_Production: 
[VIZ 1] UMAP Embedding Projection


INFO:AML_Visualizations_Production:
[VIZ 1] UMAP Embedding Projection


2026-03-06 03:21:05 [INFO] AML_Visualizations_Production:   Extracting node embeddings from trained model...


INFO:AML_Visualizations_Production:  Extracting node embeddings from trained model...


2026-03-06 03:21:30 [INFO] AML_Visualizations_Production:   Using all 111,861 validation nodes (14 sanctioned, 111,847 benign)


...

INFO:AML_Visualizations_Production:  Computing UMAP projection (this may take 3-6 minutes)...


UMAP(n_epochs=200, n_jobs=1, random_state=42, verbose=1)
Fri Mar  6 03:21:30 2026 Construct fuzzy simplicial set
Fri Mar  6 03:21:30 2026 Finding Nearest Neighbors
Fri Mar  6 03:21:30 2026 Building RP forest with 22 trees
Fri Mar  6 03:21:36 2026 NN descent for 17 iterations
	 1  /  17
	 2  /  17
	Stopping threshold met -- exiting after 2 iterations
Fri Mar  6 03:21:59 2026 Finished Nearest Neighbor Search
Fri Mar  6 03:22:02 2026 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Fri Mar  6 03:25:59 2026 Finished embedding
2026-03-06 03:26:02 [INFO] AML_Visualizations_Production:   ✓ Saved: /content/drive/MyDrive/Math_168_Folder/Data-Visualization/outputs/viz1_embedding_projection_umap.png


INFO:AML_Visualizations_Production:  ✓ Saved: /content/drive/MyDrive/Math_168_Folder/Data-Visualization/outputs/viz1_embedding_projection_umap.png


## Visualization 2: Ego-Centric Subgraphs (Production)

In [14]:
logger.info("\n[VIZ 2] Ego-Centric Subgraph Motifs")

def extract_ego_subgraph_df(df_edges: pd.DataFrame, center_node: int, radius: int = 2) -> pd.DataFrame:
    """
    Extract ego subgraph directly from DataFrame (memory-efficient).

    Uses BFS to find all nodes within radius hops from center_node,
    then filters edges to those within the ego subgraph.
    """
    visited = {center_node}
    current_level = {center_node}

    for hop in range(radius):
        next_level = set()
        edges_from = df_edges[df_edges['from_id'].isin(current_level)]
        edges_to = df_edges[df_edges['to_id'].isin(current_level)]
        next_level.update(edges_from['to_id'].values)
        next_level.update(edges_to['from_id'].values)
        next_level -= visited
        current_level = next_level
        visited.update(next_level)

    ego_edges = df_edges[
        (df_edges['from_id'].isin(visited)) &
        (df_edges['to_id'].isin(visited))
    ].copy()

    return ego_edges, visited

# Get sanctioned nodes
sanctioned_nodes = df_labels[df_labels['is_sanctioned'] == 1]['node_id'].values
logger.info(f"  Found {len(sanctioned_nodes)} sanctioned nodes")

# Extract ego subgraphs (memory-efficient)
logger.info(f"  Extracting {config.EGO_SUBGRAPH_MAX_SAMPLES} ego subgraphs directly from DataFrame...")
ego_graphs = {}
ego_node_sets = {}

for sanc_node in sanctioned_nodes[:config.EGO_SUBGRAPH_MAX_SAMPLES]:
    ego_edges_df, ego_nodes = extract_ego_subgraph_df(df_edges, sanc_node, radius=config.EGO_SUBGRAPH_RADIUS)
    ego_graphs[sanc_node] = ego_edges_df
    ego_node_sets[sanc_node] = ego_nodes
    logger.info(f"    Node {sanc_node}: {len(ego_nodes)} nodes, {len(ego_edges_df)} edges")

# Static visualizations with color gradient edges
logger.info("  Creating static visualizations...")

# Color map: light yellow (small tx) -> orange -> dark red (large tx)
edge_cmap = plt.cm.YlOrRd

MAX_VIZ_NODES = 2000  # Cap for layout computation feasibility

for idx, sanc_node in enumerate(list(ego_graphs.keys())[:config.EGO_SUBGRAPH_MAX_SAMPLES]):
    ego_edges_df = ego_graphs[sanc_node]
    ego_nodes = ego_node_sets[sanc_node]

    # Build networkx graph from ego edges
    G_ego = nx.DiGraph()
    for _, row in ego_edges_df.iterrows():
        from_id = int(row['from_id'])
        to_id = int(row['to_id'])
        weight = row['Amount Sent']
        G_ego.add_edge(from_id, to_id, weight=weight, timestamp=row['Timestamp'])

    # Cap subgraph size — keep sanctioned node + highest-degree neighbors
    if len(G_ego.nodes()) > MAX_VIZ_NODES:
        logger.info(f"    Subgraph too large ({len(G_ego.nodes())} nodes), sampling top {MAX_VIZ_NODES} by degree...")
        degrees = dict(G_ego.degree())
        degrees[sanc_node] = float('inf')  # Always keep the sanctioned node
        top_nodes = sorted(degrees, key=degrees.get, reverse=True)[:MAX_VIZ_NODES]
        G_ego = G_ego.subgraph(top_nodes).copy()
        logger.info(f"    Sampled to {len(G_ego.nodes())} nodes, {len(G_ego.edges())} edges")

    fig, ax = plt.subplots(figsize=(14, 11))

    # Spectral layout for larger graphs (cleaner structure), spring for small
    if len(G_ego.nodes()) > 500:
        logger.info(f"    Using spectral layout ({len(G_ego.nodes())} nodes)...")
        try:
            pos = nx.spectral_layout(G_ego)
        except Exception:
            logger.info("    Spectral layout failed, falling back to spring layout")
            pos = nx.spring_layout(G_ego, k=2, iterations=50, seed=42)
    else:
        pos = nx.spring_layout(G_ego, k=2, iterations=100, seed=42)

    # Draw nodes
    node_colors = ['darkred' if node == sanc_node else 'steelblue'
                   for node in G_ego.nodes()]
    node_sizes = [400 if node == sanc_node
                  else 50 + G_ego.in_degree(node) * 10
                  for node in G_ego.nodes()]

    nx.draw_networkx_nodes(G_ego, pos, node_color=node_colors, node_size=node_sizes,
                          alpha=0.85, ax=ax, edgecolors='black', linewidths=1.5)

    # Draw edges with color gradient based on transaction amount
    edges = list(G_ego.edges())
    edge_weights = np.array([G_ego[u][v]['weight'] for u, v in edges])

    # Normalize weights to [0, 1] for colormap
    if edge_weights.max() > edge_weights.min():
        log_weights = np.log1p(edge_weights)
        edge_weights_norm = (log_weights - log_weights.min()) / (log_weights.max() - log_weights.min())
    else:
        edge_weights_norm = np.ones(len(edge_weights)) * 0.5

    edge_colors = [edge_cmap(w) for w in edge_weights_norm]

    nx.draw_networkx_edges(G_ego, pos, edgelist=edges, edge_color=edge_colors,
                          arrows=True, arrowsize=10,
                          connectionstyle='arc3,rad=0.1',
                          width=0.5, alpha=0.6, ax=ax)

    # Label sanctioned node
    sanctioned_labels = {sanc_node: f"SANCTIONED\nNODE\n{sanc_node}"}
    nx.draw_networkx_labels(G_ego, pos, labels=sanctioned_labels,
                           font_size=9, font_weight='bold', ax=ax)

    ax.set_title(f'Ego-Centric Subgraph (Radius={config.EGO_SUBGRAPH_RADIUS}) Around OFAC Node {sanc_node}\n'
                f'Nodes: {len(G_ego.nodes())}, Edges: {len(G_ego.edges())}',
                fontsize=13, fontweight='bold', pad=15)
    ax.axis('off')

    # Add colorbar legend for edge colors
    sm = plt.cm.ScalarMappable(cmap=edge_cmap,
                                norm=plt.Normalize(vmin=edge_weights.min(), vmax=edge_weights.max()))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
    cbar.set_label('Transaction Amount (ETH)', fontsize=11)

    plt.tight_layout()

    output_path = config.OUTPUT_DIR / f'viz2a_ego_subgraph_static_{idx}_{sanc_node}.png'
    fig.savefig(output_path, dpi=config.DPI, bbox_inches='tight')
    logger.info(f"  ✓ Saved: {output_path}")
    plt.close()

# Interactive visualizations
logger.info("  Creating interactive visualizations (PyVis)...")
for idx, sanc_node in enumerate(list(ego_graphs.keys())[:config.EGO_SUBGRAPH_MAX_SAMPLES]):
    ego_edges_df = ego_graphs[sanc_node]
    ego_nodes = ego_node_sets[sanc_node]

    net = Network(directed=True, height='800px', width='100%', notebook=False, cdn_resources='remote')

    # Physics configuration
    options = {
        "physics": {
            "enabled": True,
            "stabilization": {
                "iterations": 200
            }
        }
    }
    net.set_options(json.dumps(options))

    for node in ego_nodes:
        n_id = int(node)  # Cast to native int for PyVis assertion check
        is_sanctioned = (node == sanc_node)
        color = 'red' if is_sanctioned else 'steelblue'
        size = 50 if is_sanctioned else 30
        title = f"{'' if is_sanctioned else ''}Node {n_id}"
        net.add_node(n_id, label=str(n_id), color=color, size=size, title=title)

    for _, row in ego_edges_df.iterrows():
        u = int(row['from_id'])
        v = int(row['to_id'])
        weight = row['Amount Sent']
        net.add_edge(u, v, value=weight, title=f"{weight:.2f} ETH")

    output_file = config.OUTPUT_DIR / f'viz2b_ego_subgraph_interactive_{idx}_{sanc_node}.html'
    net.save_graph(str(output_file))
    logger.info(f"  ✓ Saved: {output_file}")

2026-03-06 04:59:58 [INFO] AML_Visualizations_Production: 
[VIZ 2] Ego-Centric Subgraph Motifs


INFO:AML_Visualizations_Production:
[VIZ 2] Ego-Centric Subgraph Motifs


2026-03-06 04:59:58 [INFO] AML_Visualizations_Production:   Found 69 sanctioned nodes


INFO:AML_Visualizations_Production:  Found 69 sanctioned nodes


2026-03-06 04:59:58 [INFO] AML_Visualizations_Production:   Extracting 4 ego subgraphs directly from DataFrame...


...

INFO:AML_Visualizations_Production:  ✓ Saved: /content/drive/MyDrive/Math_168_Folder/Data-Visualization/outputs/viz2b_ego_subgraph_interactive_1_30069.html


2026-03-06 05:10:25 [INFO] AML_Visualizations_Production:   ✓ Saved: /content/drive/MyDrive/Math_168_Folder/Data-Visualization/outputs/viz2b_ego_subgraph_interactive_2_31407.html


INFO:AML_Visualizations_Production:  ✓ Saved: /content/drive/MyDrive/Math_168_Folder/Data-Visualization/outputs/viz2b_ego_subgraph_interactive_2_31407.html


2026-03-06 05:11:10 [INFO] AML_Visualizations_Production:   ✓ Saved: /content/drive/MyDrive/Math_168_Folder/Data-Visualization/outputs/viz2b_ego_subgraph_interactive_3_33831.html


INFO:AML_Visualizations_Production:  ✓ Saved: /content/drive/MyDrive/Math_168_Folder/Data-Visualization/outputs/viz2b_ego_subgraph_interactive_3_33831.html


## Visualization 3: Temporal Evolution Snapshots (Production)

In [16]:
logger.info("\n[VIZ 3] Temporal Evolution Snapshots")

# Create temporal windows
SECONDS_PER_WEEK = 7 * 86400
df_edges['week'] = (df_edges['Timestamp'] // SECONDS_PER_WEEK).astype(int)

weeks = sorted(df_edges['week'].unique())
window_indices = np.linspace(0, len(weeks) - 1, config.TEMPORAL_WINDOWS, dtype=int)
selected_weeks = [weeks[i] for i in window_indices]

logger.info(f"  Total weeks: {len(weeks)}, Selected: {config.TEMPORAL_WINDOWS}")

fig, axes = plt.subplots(2, 5, figsize=(26, 11))
axes = axes.flatten()

sanctioned_set = set(df_labels[df_labels['is_sanctioned'] == 1]['node_id'].values)

MAX_WEEK_EDGES = 50000  # Cap per weekly snapshot

for plot_idx, week in enumerate(selected_weeks):
    ax = axes[plot_idx]

    week_edges = df_edges[df_edges['week'] == week]
    logger.info(f"  Processing week {week} ({plot_idx + 1}/{config.TEMPORAL_WINDOWS}): {len(week_edges):,} edges")

    if len(week_edges) == 0:
        ax.text(0.5, 0.5, 'No transactions', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Week {week}\n(0 edges)')
        ax.axis('off')
        continue

    # Sample if too many edges
    if len(week_edges) > MAX_WEEK_EDGES:
        logger.info(f"    Sampling {MAX_WEEK_EDGES:,} of {len(week_edges):,} edges...")
        week_edges = week_edges.sample(n=MAX_WEEK_EDGES, random_state=42)

    # Build graph using vectorized pandas method (not iterrows)
    G_week = nx.from_pandas_edgelist(
        week_edges, source='from_id', target='to_id',
        edge_attr='Amount Sent', create_using=nx.DiGraph()
    )

    if len(G_week.nodes()) > 0:
        largest_cc = max(nx.weakly_connected_components(G_week), key=len)
        G_week_viz = G_week.subgraph(largest_cc).copy()
    else:
        G_week_viz = G_week

    pos = nx.spring_layout(G_week_viz, k=0.5, iterations=20, seed=42)

    node_sizes = [300 if node in sanctioned_set else (10 + G_week_viz.degree(node) * 5)
                  for node in G_week_viz.nodes()]
    node_colors = ['darkred' if node in sanctioned_set else 'steelblue'
                   for node in G_week_viz.nodes()]

    nx.draw_networkx_nodes(G_week_viz, pos, node_size=node_sizes, node_color=node_colors,
                          alpha=0.7, ax=ax)
    nx.draw_networkx_edges(G_week_viz, pos, edge_color='gray', arrows=False, alpha=0.2, ax=ax)

    ax.set_title(f'Week {week}\n({len(G_week_viz.nodes())} nodes, {len(G_week_viz.edges())} edges)',
                fontsize=10, fontweight='bold')
    ax.axis('off')

for idx in range(len(selected_weeks), len(axes)):
    fig.delaxes(axes[idx])

fig.suptitle('Temporal Evolution of Transaction Graph (10 snapshots)\n'
            'Red = Sanctioned addresses; Blue = Benign addresses; Size proportional to Degree',
            fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout()
output_path = config.OUTPUT_DIR / 'viz3_temporal_evolution_grid.png'
fig.savefig(output_path, dpi=config.DPI, bbox_inches='tight')
logger.info(f"  ✓ Saved: {output_path}")
plt.close()

2026-03-06 05:29:08 [INFO] AML_Visualizations_Production: 
[VIZ 3] Temporal Evolution Snapshots


INFO:AML_Visualizations_Production:
[VIZ 3] Temporal Evolution Snapshots


2026-03-06 05:29:09 [INFO] AML_Visualizations_Production:   Total weeks: 550, Selected: 10


INFO:AML_Visualizations_Production:  Total weeks: 550, Selected: 10


2026-03-06 05:29:09 [INFO] AML_Visualizations_Production:   Processing week 0 (1/10): 1,505 edges


...

INFO:AML_Visualizations_Production:  Processing week 488 (9/10): 21,282 edges


2026-03-06 06:48:36 [INFO] AML_Visualizations_Production:   Processing week 549 (10/10): 8,607 edges


INFO:AML_Visualizations_Production:  Processing week 549 (10/10): 8,607 edges


2026-03-06 06:49:02 [INFO] AML_Visualizations_Production:   ✓ Saved: /content/drive/MyDrive/Math_168_Folder/Data-Visualization/outputs/viz3_temporal_evolution_grid.png


INFO:AML_Visualizations_Production:  ✓ Saved: /content/drive/MyDrive/Math_168_Folder/Data-Visualization/outputs/viz3_temporal_evolution_grid.png


## Visualization 4: Feature Importance & Model Decisions

In [ ]:
logger.info("\n[VIZ 4a] SHAP Feature Importance (Edge Feature Analysis)")

from sklearn.ensemble import GradientBoostingRegressor

# Exclude leaked/ID features: Is Laundering (ground-truth label) and EdgeID (arbitrary index)
all_edge_feat_cols = ['EdgeID', 'Timestamp', 'Amount Sent', 'Sent Currency',
                      'Amount Received', 'Received Currency', 'Payment Format', 'Is Laundering']
EXCLUDE_FEATURES = {'EdgeID', 'Is Laundering'}
edge_feat_cols = [c for c in all_edge_feat_cols if c not in EXCLUDE_FEATURES]
feature_display_names = ['Timestamp', 'Amount Sent', 'Sent Currency',
                         'Amount Received', 'Received Currency', 'Payment Format']

logger.info(f"  Using {len(edge_feat_cols)} features (excluded: {EXCLUDE_FEATURES})")

# Aggregate edge features per node (mean of incident edges)
logger.info("  Aggregating edge features per node...")
out_agg = df_edges.groupby('from_id')[edge_feat_cols].mean()
in_agg = df_edges.groupby('to_id')[edge_feat_cols].mean()

max_nid = data.num_nodes
X_all = np.zeros((max_nid, len(edge_feat_cols)))
node_count = np.zeros(max_nid)

out_ids = out_agg.index.values.astype(int)
valid_out = out_ids < max_nid
X_all[out_ids[valid_out]] += out_agg.values[valid_out]
node_count[out_ids[valid_out]] += 1

in_ids = in_agg.index.values.astype(int)
valid_in = in_ids < max_nid
X_all[in_ids[valid_in]] += in_agg.values[valid_in]
node_count[in_ids[valid_in]] += 1

node_count[node_count == 0] = 1
X_all /= node_count[:, np.newaxis]
logger.info(f"  Aggregated features for {(node_count > 0).sum():,} nodes")

# Get GNN predicted probabilities
logger.info("  Computing GNN predictions...")
with torch.no_grad():
    logits = model(data.x, data.edge_index, data.edge_attr)
    y_proba = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

# Fit surrogate: aggregated edge features → GNN probability
val_idx = torch.nonzero(data.val_mask, as_tuple=True)[0].cpu().numpy()
X_val = X_all[val_idx]
y_val = y_proba[val_idx]

logger.info("  Fitting surrogate model (GBR)...")
surrogate = GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)
surrogate.fit(X_val, y_val)

from sklearn.metrics import r2_score
r2 = r2_score(y_val, surrogate.predict(X_val))
logger.info(f"  Surrogate R² on validation set: {r2:.4f}")

# SHAP: explain sanctioned nodes + sample of benign
logger.info("  Computing SHAP values...")
explainer = shap.TreeExplainer(surrogate)

sanctioned_idx = np.where(data.y.cpu().numpy() == 1)[0]
sanc_in_val = np.intersect1d(sanctioned_idx, val_idx)
benign_in_val = np.setdiff1d(val_idx, sanctioned_idx)
np.random.seed(42)
benign_sample = np.random.choice(benign_in_val,
                                  min(config.SHAP_SAMPLE_SIZE, len(benign_in_val)),
                                  replace=False)
explain_idx = np.concatenate([sanc_in_val, benign_sample])
X_explain = X_all[explain_idx]

shap_values = explainer.shap_values(X_explain)
logger.info(f"  SHAP computed for {len(explain_idx)} nodes ({len(sanc_in_val)} sanctioned + {len(benign_sample)} benign)")

# SHAP bar chart
logger.info("  Creating SHAP feature importance plot...")
shap.summary_plot(shap_values, X_explain, feature_names=feature_display_names,
                  show=False, plot_type='bar')
fig = plt.gcf()
fig.set_size_inches(12, 8)
plt.title('SHAP Feature Importance: Edge Feature Contributions to Sanctions Prediction\n'
          f'(GBR surrogate model, R\u00b2={r2:.3f}, {len(explain_idx)} nodes, excluding leaked/ID features)',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
path = config.OUTPUT_DIR / 'viz4a_shap_feature_importance.png'
plt.savefig(path, dpi=config.DPI, bbox_inches='tight')
logger.info(f"  \u2713 Saved: {path}")
plt.close('all')

In [ ]:
logger.info("\n[VIZ 4b] Gradient-based Edge Feature Attribution")

# Edge feature order in edge_attr tensor: [EdgeID(0), Timestamp(1), Amount Sent(2),
# Sent Currency(3), Amount Received(4), Received Currency(5), Payment Format(6), Is Laundering(7)]
# Exclude index 0 (EdgeID) and index 7 (Is Laundering)
KEEP_INDICES = [1, 2, 3, 4, 5, 6]  # Timestamp, Amount Sent, Sent Currency, Amount Received, Received Currency, Payment Format
feature_display_names = ['Timestamp', 'Amount Sent', 'Sent Currency',
                         'Amount Received', 'Received Currency', 'Payment Format']

# Freeze model params (only need gradients w.r.t. edge_attr input)
for param in model.parameters():
    param.requires_grad_(False)

logger.info("  Computing gradients w.r.t. edge features...")
edge_attr_input = data.edge_attr.detach().clone().requires_grad_(True)

logits = model(data.x, data.edge_index, edge_attr_input)

# Backprop from sanctioned-class logits for sanctioned nodes
sanctioned_idx = torch.nonzero(data.y == 1, as_tuple=True)[0]
target = logits[sanctioned_idx, 1].sum()
target.backward()

grads = edge_attr_input.grad  # (num_edges, 8)

# Select only the 6 legitimate feature dimensions
grads_filtered = grads[:, KEEP_INDICES]  # (num_edges, 6)

# Global mean |gradient| per feature
global_importance = grads_filtered.abs().mean(dim=0).cpu().numpy()

# Sanctioned-adjacent edges only (edges touching at least one sanctioned node)
sanc_tensor = torch.zeros(data.num_nodes, dtype=torch.bool, device=device)
sanc_tensor[sanctioned_idx] = True
src, dst = data.edge_index
sanc_mask = sanc_tensor[src] | sanc_tensor[dst]
sanc_importance = grads_filtered[sanc_mask].abs().mean(dim=0).cpu().numpy()

logger.info(f"  Total edges: {len(grads):,}, Sanctioned-adjacent: {sanc_mask.sum().item():,}")
logger.info(f"  Using {len(KEEP_INDICES)} features (excluded: EdgeID, Is Laundering)")

# Restore model params
for param in model.parameters():
    param.requires_grad_(True)
edge_attr_input.grad = None

# Plot dual bar charts
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

global_order = np.argsort(global_importance)
sanc_order = np.argsort(sanc_importance)

axes[0].barh(range(len(feature_display_names)), global_importance[global_order],
             color='steelblue', alpha=0.85)
axes[0].set_yticks(range(len(feature_display_names)))
axes[0].set_yticklabels([feature_display_names[i] for i in global_order], fontsize=11)
axes[0].set_xlabel('Mean |Gradient|', fontsize=12, fontweight='bold')
axes[0].set_title('Global Edge Feature Sensitivity\n(All edges in graph)',
                  fontsize=13, fontweight='bold')

axes[1].barh(range(len(feature_display_names)), sanc_importance[sanc_order],
             color='darkred', alpha=0.85)
axes[1].set_yticks(range(len(feature_display_names)))
axes[1].set_yticklabels([feature_display_names[i] for i in sanc_order], fontsize=11)
axes[1].set_xlabel('Mean |Gradient|', fontsize=12, fontweight='bold')
axes[1].set_title('Sanctioned Node Edge Feature Sensitivity\n(Edges incident to OFAC-sanctioned nodes)',
                  fontsize=13, fontweight='bold')

fig.suptitle('Gradient-Based Feature Attribution: Model Sensitivity to Edge Features\n(excluding leaked/ID features)',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
path = config.OUTPUT_DIR / 'viz4b_gradient_attribution.png'
fig.savefig(path, dpi=config.DPI, bbox_inches='tight')
logger.info(f"  \u2713 Saved: {path}")
plt.close('all')

## Visualization 5: Evaluation Metrics (Production)

In [19]:
logger.info("\n[VIZ 5] Evaluation Metrics (Confusion Matrix & Curves)")

# Get validation predictions
logger.info("  Computing validation predictions...")
with torch.no_grad():
    logits = model(data.x, data.edge_index, data.edge_attr)
    val_logits = logits[data.val_mask]

y_true = data.y[data.val_mask].cpu().numpy()
y_proba = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
y_pred = (y_proba >= 0.5).astype(int)

logger.info(f"  Predictions: {(y_pred == 1).sum()} positive, {(y_pred == 0).sum()} negative")

# Confusion Matrix
logger.info("  Creating confusion matrix...")
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Sanctioned', 'Sanctioned'],
            yticklabels=['Non-Sanctioned', 'Sanctioned'],
            cbar_kws={'label': 'Count'},
            annot_kws={'size': 14, 'weight': 'bold'}, ax=ax)
ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix (Validation Set)\nThreshold = 0.5',
            fontsize=13, fontweight='bold', pad=15)

tn, fp, fn, tp = cm.ravel()
metrics_text = f'TP={tp}, FN={fn}\nFP={fp}, TN={tn}\nPrec={tp/(tp+fp) if tp+fp>0 else 0:.3f}, Rec={tp/(tp+fn) if tp+fn>0 else 0:.3f}'
ax.text(0.5, -0.18, metrics_text, transform=ax.transAxes, ha='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
path = config.OUTPUT_DIR / 'viz5a_confusion_matrix.png'
fig.savefig(path, dpi=config.DPI, bbox_inches='tight')
logger.info(f"  ✓ Saved: {path}")
plt.close()

# ROC Curve
logger.info("  Creating ROC curve...")
fpr, tpr, _ = roc_curve(y_true, y_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(fpr, tpr, color='darkorange', lw=3, label=f'ROC Curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
ax.fill_between(fpr, tpr, alpha=0.2, color='darkorange')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('ROC Curve', fontsize=13, fontweight='bold', pad=15)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
path = config.OUTPUT_DIR / 'viz5b_roc_curve.png'
fig.savefig(path, dpi=config.DPI, bbox_inches='tight')
logger.info(f"  ✓ Saved: {path}")
plt.close()

# PR Curve
logger.info("  Creating Precision-Recall curve...")
precision, recall, _ = precision_recall_curve(y_true, y_proba)
pr_auc = auc(recall, precision)
baseline_precision = (y_true == 1).sum() / len(y_true)

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(recall, precision, color='green', lw=3, label=f'PR Curve (AUC = {pr_auc:.3f})')
ax.fill_between(recall, precision, alpha=0.2, color='green')
ax.axhline(y=baseline_precision, color='red', linestyle='--', lw=2,
          label=f'Baseline ({baseline_precision:.3f})')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
ax.set_title('Precision-Recall Curve (Primary for Imbalanced Data)',
            fontsize=13, fontweight='bold', pad=15)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
path = config.OUTPUT_DIR / 'viz5c_precision_recall_curve.png'
fig.savefig(path, dpi=config.DPI, bbox_inches='tight')
logger.info(f"  ✓ Saved: {path}")
plt.close()

2026-03-06 07:02:45 [INFO] AML_Visualizations_Production: 
[VIZ 5] Evaluation Metrics (Confusion Matrix & Curves)


INFO:AML_Visualizations_Production:
[VIZ 5] Evaluation Metrics (Confusion Matrix & Curves)


2026-03-06 07:02:45 [INFO] AML_Visualizations_Production:   Computing validation predictions...


INFO:AML_Visualizations_Production:  Computing validation predictions...


2026-03-06 07:03:06 [INFO] AML_Visualizations_Production:   Predictions: 44 positive, 111817 negative


...

INFO:AML_Visualizations_Production:  ✓ Saved: /content/drive/MyDrive/Math_168_Folder/Data-Visualization/outputs/viz5b_roc_curve.png


2026-03-06 07:03:07 [INFO] AML_Visualizations_Production:   Creating Precision-Recall curve...


INFO:AML_Visualizations_Production:  Creating Precision-Recall curve...


2026-03-06 07:03:08 [INFO] AML_Visualizations_Production:   ✓ Saved: /content/drive/MyDrive/Math_168_Folder/Data-Visualization/outputs/viz5c_precision_recall_curve.png


INFO:AML_Visualizations_Production:  ✓ Saved: /content/drive/MyDrive/Math_168_Folder/Data-Visualization/outputs/viz5c_precision_recall_curve.png


## Completion Summary

In [20]:
logger.info("\n" + "="*80)
logger.info("VISUALIZATIONS COMPLETE!")
logger.info("="*80)
logger.info(f"\nOutput Directory: {config.OUTPUT_DIR}")
logger.info("\nGenerated Files:")
logger.info("  ✓ viz1_embedding_projection_umap.png")
logger.info("  ✓ viz2a_ego_subgraph_static_*.png")
logger.info("  ✓ viz2b_ego_subgraph_interactive_*.html")
logger.info("  ✓ viz3_temporal_evolution_grid.png")
logger.info("  ✓ viz5a_confusion_matrix.png")
logger.info("  ✓ viz5b_roc_curve.png")
logger.info("  ✓ viz5c_precision_recall_curve.png")
logger.info("\n" + "="*80)

2026-03-06 07:06:10 [INFO] AML_Visualizations_Production: 


INFO:AML_Visualizations_Production:


2026-03-06 07:06:10 [INFO] AML_Visualizations_Production: VISUALIZATIONS COMPLETE!


INFO:AML_Visualizations_Production:VISUALIZATIONS COMPLETE!


2026-03-06 07:06:10 [INFO] AML_Visualizations_Production: ================================================================================


...

INFO:AML_Visualizations_Production:  ✓ viz5b_roc_curve.png


2026-03-06 07:06:11 [INFO] AML_Visualizations_Production:   ✓ viz5c_precision_recall_curve.png


INFO:AML_Visualizations_Production:  ✓ viz5c_precision_recall_curve.png


2026-03-06 07:06:11 [INFO] AML_Visualizations_Production: 


INFO:AML_Visualizations_Production:
